# 01 — Train a base model

Basic guide for training a base model. 
The original repository contains a CLI commands for usage and training. However we introduced and built a few more classes to aid in training and usage of the model, so that it is more user friendly but also customisable. 

In general there are a few key classes that are important for basic usage 
1) Dataloader (converts ASE atoms objects to pytorch dataloaders)
2) Trainer (settles the training, weight decay, checkpointing etc)
3) Model factory (function that helps to initialise model, with foundation weights)
4) Tester (tests the model based on various metrics)

There is also a key Strategies class, but that is covered separately in the second tutorial notebook, mainly used to customise the model specifically for transfer learning. 

Note that in this notebook the cells won't actually compile since the data filepaths are dummy file paths, not actual valid ones. But they contain valid syntax that will run if specified properly. 

## 1. Imports and basic settings

In [ ]:
import json
import random
from pathlib import Path

import numpy as np
import torch
import ase.io 

from mace.data.atom_data_loader import AtomDataLoaderBuilder
from mace.modules.loss import InvariantsWeightedEnergyForcesNacsDipoleLoss
from mace.training.model_factory import initialise_autoencoder
from mace.training.trainer import Trainer

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_default_dtype(torch.float64)

device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Read training and validation data

We start with an input of extended xyz files. This file will be read by ASE to form ASE atoms objects. This objects are used as the input for the dataloader later on 

For each geometry's xyz `atoms.info`, the following info is required: 

- `REF_energy`: `[states]`.
- `REF_forces`: `[atoms, states, 3]`.
- `REF_nacs`: `[atoms, pairs, 3]`, containing **raw NACs**.

NAC pairs are ordered `(0,1), (0,2), (1,2)`
Do **NOT** specify smooth NACs in the xyz file, that conversion is handeled internally. 

Units expected are Angstrom for distance and eV for energies for the xyz file. (By extension eV/A for forces, A-1 for raw NACs)

In [ ]:
data_file_path = "<insert training file path>"
valid_file_path = "<insert validation file path>"

train_atoms = ase.io.read(data_file_path, index=":")
valid_atoms = ase.io.read(valid_file_path, index=":")

## 3. Create the data loader builder

First we define a class called `AtomDataLoaderBuilder`. This builder is responsible for converting ASE atoms objects into the pytorch data loaders. Internally, it handles the graph conversions, labels readings, baseline energy levels, metadata, smooth/raw NAC conversions etc. 
The builder is initialised with parameters related to graph construction, and returns a dataloader when called with `load()` in the next step.

`cutoff=5.0` sets the neighbour distance limit in the same length units as the positions. The three label keys tell the builder where to find energies, forces, and raw NACs in `atoms.info`. These keys should match whatever is inside the xyz file

`E0s` estimates per-element atomic energy baselines from the training data. We specify this is a dictionary of {element: energies}

In [ ]:
data_builder = AtomDataLoaderBuilder(
    cutoff=5.0, 
    energy_key="REF_energy", 
    forces_key="REF_forces", 
    nacs_key="REF_nacs",
    E0s= {"H": -15.357929595328724, "C": -1032.083979117871},
)


## 4. Build loaders and obtain metadata

When using the data builder class, we can return a pytorch dataloader object with the `load()`function. Here, we input the ASE atoms object from before, the desired batch size, and a dataloader is returned. 

`batch_size=10` refers to up to ten complete geometries per batch, not ten atoms. Training data is shuffled using our seed; validation data is not shuffled. 

`get_metadata()` returns the dataset information needed by the model: element identities (`atomic_numbers`), E0s (`atomic_energies`), cutoff (`r_max`), state count (`n_energies`), and neighbour statistics (`avg_num_neighbors`). 

This metadata is super important because its also used to initialise the model. Between train/validation, the same dataloader should be used so that they share the same information (eg baseline energies). 

In [ ]:
train_loader = data_builder.load(
    train_atoms, 
    batch_size=10, 
    shuffle=True, 
    seed=seed
)

valid_loader = data_builder.load(
    valid_atoms, 
    batch_size=10, 
    shuffle=False
)

metadata = data_builder.get_metadata()

## 5. Initialise a fresh model


In [ ]:
num_states = metadata.n_energies
num_pairs = num_states * (num_states - 1) // 2

model_settings = {
    "preset": "default_ani",
    "latent_dim": 16,
    "energy_decoder": "matrix",
    "compute_nacs": True,
    "nac_num": num_pairs,
    "load_base": None,
}
model = initialise_autoencoder(metadata, **model_settings)

## 6. Define the loss


In [ ]:
loss_fn = InvariantsWeightedEnergyForcesNacsDipoleLoss(
    energy_weight=1.0, forces_weight=1.0, nacs_weight=1.0,
).to(device)

## 7. Configure Trainer


In [ ]:
trainer = Trainer(
    device=device, max_epochs=100, optimiser_lr=1e-3,
    ema_decay=0.99, restore_best=True,
)

## 8. Choose an output directory

## 9. Train the model


In [ ]:
model, history = trainer.train_model(
    model, train_loader, valid_loader, loss_fn,
    checkpoint_epoch=10, checkpoint_models_dir=output_dir / "checkpoints",
    compute_nacs=True,
)

## 10. Inspect results and save the best weights

In [ ]:
if history["best_epoch"] == 0:
    raise RuntimeError("Training produced no valid best model.")

best_index = history["epoch"].index(history["best_epoch"])
print("Best epoch:", history["best_epoch"])
print("Energy MAE:", history["valid_energy_mae"][best_index])
print("Force MAE:", history["valid_force_mae"][best_index])
print("Smooth NAC MAE:", history["valid_smooth_nac_phase_mae"][best_index])
print("Raw NAC MAE:", history["valid_raw_nac_phase_mae"][best_index])

torch.save(model.state_dict(), output_dir / "best_model.pt")
(output_dir / "history.json").write_text(json.dumps(history, indent=2))